# CV_PR3 — Computer Vision Project
**Red & White Skill Education | Deep Learning | PR 3**

This notebook covers five tasks: (1) Morphological Operations, (2) Bitwise Operations & Histograms, (3) YuNet Face Detection, (4) YOLOv8 Object Detection, and (5) an Integrated Pipeline.

Images are in `data/images/`, model weights in `data/models/`, plots saved to `plots/`.

## Setup — Dependencies
Install: `pip install opencv-python==5.0.0.* opencv-contrib-python==5.0.0.* ultralytics matplotlib numpy`

In [ ]:
# Uncomment to install:
# !pip install opencv-python==5.0.0.* opencv-contrib-python==5.0.0.* ultralytics matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')

# Confirm versions
print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)

# Directories
IMG_DIR   = "data/images"
MODEL_DIR = "data/models"
PLOT_DIR  = "plots"
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Helper: display image inline (BGR→RGB)
def show(img, title="", cmap=None):
    plt.figure(figsize=(5, 4))
    if img.ndim == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title); plt.axis('off'); plt.tight_layout(); plt.show()

def show_row(images, titles, figsize=(14, 4), save_as=None):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1: axes = [axes]
    for ax, img, t in zip(axes, images, titles):
        if img.ndim == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap='gray')
        ax.set_title(t, fontsize=9); ax.axis('off')
    plt.tight_layout()
    if save_as:
        plt.savefig(os.path.join(PLOT_DIR, save_as), dpi=120, bbox_inches='tight')
    plt.show()

# Load sample images
colour_img  = cv2.imread(os.path.join(IMG_DIR, "animal_dog.jpg"))
gray_img    = cv2.cvtColor(colour_img, cv2.COLOR_BGR2GRAY)
morph_img   = cv2.imread(os.path.join(IMG_DIR, "nature_flower.jpg"))
face_img1   = cv2.imread(os.path.join(IMG_DIR, "person1.jpg"))
face_img2   = cv2.imread(os.path.join(IMG_DIR, "person2.jpg"))
yolo_img    = cv2.imread(os.path.join(IMG_DIR, "city_street.jpg"))

print(f"Colour shape/dtype: {colour_img.shape} {colour_img.dtype}")
print(f"Grayscale shape/dtype: {gray_img.shape} {gray_img.dtype}")

show_row([colour_img, gray_img], ["Colour (BGR)", "Grayscale"], figsize=(10,4))

## Task 1: Environment Setup, Image I/O & Morphological Operations

**Objective:** Understand and apply the four morphological operations (erosion, dilation, opening, closing) to binary images, and explore how kernel shape and size affect the result.

**Expected output:** 1×5 morphology grid, 3×3 kernel comparison grid.

In [ ]:
# 1.2 Binarize a test image
morph_gray = cv2.cvtColor(morph_img, cv2.COLOR_BGR2GRAY)

# Add salt-and-pepper noise for better demo
noisy = morph_gray.copy().astype(np.float32)
noise = np.random.randint(0, 2, noisy.shape)
noisy[noise == 0] = 0
noisy[noise == 1] = 255
noisy = np.clip(noisy, 0, 255).astype(np.uint8)

_, binary = cv2.threshold(noisy, 127, 255, cv2.THRESH_BINARY)
show_row([morph_gray, noisy, binary],
         ["Original Grayscale", "Noisy", "Binary (thresh=127)"],
         figsize=(14, 4))

In [ ]:
# 1.3 Four morphological operations
kernel = np.ones((5, 5), np.uint8)

eroded  = cv2.erode(binary, kernel)
dilated = cv2.dilate(binary, kernel)
opened  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  kernel)
closed  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

show_row(
    [binary, eroded, dilated, opened, closed],
    ["Original Binary", "Erosion", "Dilation", "Opening", "Closing"],
    figsize=(18, 4), save_as="morphology_grid.png"
)

In [ ]:
# 1.4 Kernel shape × size comparison (3×3 grid)
shapes = [
    (cv2.MORPH_RECT,    "Rect"),
    (cv2.MORPH_ELLIPSE, "Ellipse"),
    (cv2.MORPH_CROSS,   "Cross"),
]
sizes = [3, 5, 9]

fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for row, sz in enumerate(sizes):
    for col, (shape_code, shape_name) in enumerate(shapes):
        k = cv2.getStructuringElement(shape_code, (sz, sz))
        result = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k)
        axes[row, col].imshow(result, cmap='gray')
        axes[row, col].set_title(f"{shape_name} {sz}×{sz}", fontsize=9)
        axes[row, col].axis('off')
plt.suptitle("Kernel Shape × Size Comparison (Opening)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "kernel_comparison.png"), dpi=120, bbox_inches='tight')
plt.show()
print("Best for removing salt-and-pepper noise: Ellipse 5×5 — preserves object outline well.")

### 1.5 Why Morphology Works

| Operation | Mechanism | Real-world Use |
|-----------|-----------|----------------|
| **Erosion** | Local *minimum* filter — shrinks white regions, removes small bright noise pixels | Remove small dust/speckles in a scanned document before OCR |
| **Dilation** | Local *maximum* filter — grows white regions, fills small dark gaps | Fill gaps in detected road markings before lane-tracking |
| **Opening** (Erosion → Dilation) | Removes small noise while preserving overall shape/size | Clean up a motion-detection mask before sending to a face detector |
| **Closing** (Dilation → Erosion) | Fills small holes without shrinking the object | Fill holes in a segmented silhouette before measuring object area |

**Rule of thumb:** Use *opening* before a downstream detector to reduce false detections from noise. Use *closing* after thresholding to make segmentation masks solid before contour extraction.

## Task 2: Bitwise Operations & Image Histograms

**Objective:** Apply AND/OR/XOR/NOT bitwise operations between binary masks; use masking for real-photo ROI extraction; compute and compare grayscale and colour histograms; adjust brightness and contrast.

**Expected output:** 2×3 bitwise grid, histogram plots, brightness/contrast comparison.

In [ ]:
# 2.1 Bitwise AND / OR / XOR / NOT
h, w = 300, 300
mask1 = np.zeros((h, w), np.uint8)
mask2 = np.zeros((h, w), np.uint8)

cv2.circle(mask1, (120, 150), 100, 255, -1)        # circle
cv2.rectangle(mask2, (100, 80), (260, 220), 255, -1) # rectangle

b_and = cv2.bitwise_and(mask1, mask2)
b_or  = cv2.bitwise_or (mask1, mask2)
b_xor = cv2.bitwise_xor(mask1, mask2)
b_not = cv2.bitwise_not(mask1)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, img, t in zip(axes.flat,
                      [mask1, mask2, b_and, b_or, b_xor, b_not],
                      ["Mask 1 (Circle)", "Mask 2 (Rect)", "AND", "OR", "XOR", "NOT(Mask1)"]):
    ax.imshow(img, cmap='gray'); ax.set_title(t, fontsize=11); ax.axis('off')
plt.suptitle("Bitwise Operations", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "bitwise_grid.png"), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 2.2 Applied use-case: masked overlay
photo = cv2.imread(os.path.join(IMG_DIR, "nature_flower.jpg"))
ph, pw = photo.shape[:2]

# Circular mask same size as photo
circle_mask = np.zeros((ph, pw), np.uint8)
cv2.circle(circle_mask, (pw//2, ph//2), min(ph, pw)//3, 255, -1)

# Extract circular ROI
extracted = cv2.bitwise_and(photo, photo, mask=circle_mask)

# Rectangular watermark composite (logo box)
logo_overlay = photo.copy()
rect_mask2 = np.zeros((ph, pw), np.uint8)
cv2.rectangle(rect_mask2, (10, 10), (200, 60), 255, -1)
logo_region = np.zeros_like(photo)
cv2.putText(logo_region, "CV_PR3", (15, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,255), 2)
watermarked = cv2.bitwise_or(logo_overlay, logo_region)

show_row([photo, extracted, watermarked],
         ["Original", "Circular ROI (bitwise_and)", "Watermark Overlay (bitwise_or)"],
         figsize=(15, 5))

In [ ]:
# 2.3 Grayscale & per-channel colour histograms
hist_img  = cv2.imread(os.path.join(IMG_DIR, "animal_dog.jpg"))
hist_gray = cv2.cvtColor(hist_img, cv2.COLOR_BGR2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grayscale histogram
gray_hist = cv2.calcHist([hist_gray], [0], None, [256], [0, 256])
axes[0].plot(gray_hist, color='black')
axes[0].set_title("Grayscale Histogram"); axes[0].set_xlabel("Pixel Value"); axes[0].set_ylabel("Count")
axes[0].set_xlim([0, 256])

# Colour histogram (B, G, R)
colors = ('blue', 'green', 'red')
for i, col in enumerate(colors):
    ch_hist = cv2.calcHist([hist_img], [i], None, [256], [0, 256])
    axes[1].plot(ch_hist, color=col, label=col.upper())
axes[1].set_title("BGR Colour Histogram"); axes[1].set_xlabel("Pixel Value")
axes[1].set_ylabel("Count"); axes[1].legend(); axes[1].set_xlim([0, 256])

plt.suptitle("Image Histograms", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "histogram_comparison.png"), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 2.4 Brightness & contrast adjustment
orig    = cv2.imread(os.path.join(IMG_DIR, "animal_dog.jpg"))
bright  = cv2.convertScaleAbs(orig, alpha=1.5, beta=30)    # higher contrast + brighter
dark    = cv2.convertScaleAbs(orig, alpha=0.7, beta=-30)   # lower contrast + darker

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, img, t in zip(axes[0],
                      [orig, bright, dark],
                      ["Original", "Bright (α=1.5, β=30)", "Dark (α=0.7, β=-30)"]):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_title(t, fontsize=10); ax.axis('off')

for ax, img, t, col in zip(axes[1],
                            [orig, bright, dark],
                            ["Hist: Original", "Hist: Bright", "Hist: Dark"],
                            ['black', 'orange', 'blue']):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h = cv2.calcHist([g], [0], None, [256], [0, 256])
    ax.plot(h, color=col); ax.set_title(t, fontsize=10)
    ax.set_xlim([0, 256]); ax.set_xlabel("Pixel Value"); ax.set_ylabel("Count")

plt.suptitle("Brightness & Contrast Adjustment", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "brightness_contrast.png"), dpi=120, bbox_inches='tight')
plt.show()

### 2.5 Alpha & Beta — What They Control

**Formula:** `output = α × input + β`  (pixel values clipped to [0, 255])

| Parameter | Effect | Typical Value |
|-----------|--------|---------------|
| **α (alpha) > 1** | Increases contrast (stretches histogram) | 1.5–2.0 |
| **α (alpha) < 1** | Decreases contrast (compresses histogram) | 0.5–0.9 |
| **β (beta) > 0** | Increases brightness (shifts histogram right) | 30–80 |
| **β (beta) < 0** | Decreases brightness (shifts histogram left) | −30 to −80 |

Use `convertScaleAbs` with `alpha=1.0, beta=0` to leave the image unchanged as a sanity check.

## Task 3: Face Detection with YuNet (OpenCV 5.0)

**Objective:** Use OpenCV's built-in YuNet CNN-based face detector (`cv2.FaceDetectorYN`) on static images. Visualise confidence scores and 5-point facial landmarks. Experiment with score thresholds. Demonstrate face blurring/anonymisation.

**Expected output:** Annotated face detections, threshold experiment, blurred faces.

In [ ]:
# 3.1 Download YuNet model weights
import urllib.request

yunet_url  = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"
yunet_path = os.path.join(MODEL_DIR, "face_detection_yunet_2023mar.onnx")

if not os.path.exists(yunet_path):
    print("Downloading YuNet weights...")
    urllib.request.urlretrieve(yunet_url, yunet_path)
    print("Downloaded:", yunet_path)
else:
    print("YuNet model already exists:", yunet_path)

print("File size:", os.path.getsize(yunet_path), "bytes")

In [ ]:
# 3.2 Face detection function with landmarks & confidence
def detect_faces(img, model_path, score_thresh=0.7, nms_thresh=0.3):
    h, w = img.shape[:2]
    detector = cv2.FaceDetectorYN.create(
        model_path, "", (w, h),
        score_threshold=score_thresh,
        nms_threshold=nms_thresh
    )
    _, faces = detector.detect(img)
    return faces  # may be None if no faces found

def draw_faces(img, faces, color=(0, 255, 0)):
    out = img.copy()
    if faces is None:
        return out
    for face in faces:
        x, y, fw, fh, conf = int(face[0]), int(face[1]), int(face[2]), int(face[3]), face[14]
        cv2.rectangle(out, (x, y), (x+fw, y+fh), color, 2)
        cv2.putText(out, f"{conf:.2f}", (x, y-6), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 1)
        # Draw 5 landmarks: right eye, left eye, nose tip, right mouth, left mouth
        lm_colors = [(255,0,0),(0,0,255),(0,255,255),(255,0,255),(0,165,255)]
        for i, lc in enumerate(lm_colors):
            lx, ly = int(face[4 + i*2]), int(face[5 + i*2])
            cv2.circle(out, (lx, ly), 3, lc, -1)
    return out

# Run on static images
for fname in ["person1.jpg", "person2.jpg", "person3.jpg"]:
    img = cv2.imread(os.path.join(IMG_DIR, fname))
    faces = detect_faces(img, yunet_path, score_thresh=0.7)
    annotated = draw_faces(img, faces)
    n = len(faces) if faces is not None else 0
    show(annotated, f"{fname} — {n} face(s) detected (thresh=0.7)")

# Save one result
result_face = cv2.imread(os.path.join(IMG_DIR, "person2.jpg"))
faces2 = detect_faces(result_face, yunet_path)
ann2 = draw_faces(result_face, faces2)
cv2.imwrite(os.path.join(PLOT_DIR, "face_detection_comparison.png"), ann2)
print("Saved face_detection_comparison.png")

In [ ]:
# 3.3 Score-threshold experiment: 0.5 / 0.7 / 0.9
test_img = cv2.imread(os.path.join(IMG_DIR, "person2.jpg"))
thresholds = [0.5, 0.7, 0.9]
results, titles = [], []

for t in thresholds:
    faces = detect_faces(test_img, yunet_path, score_thresh=t)
    n = len(faces) if faces is not None else 0
    ann = draw_faces(test_img, faces)
    results.append(ann)
    titles.append(f"thresh={t}  ({n} face{'s' if n!=1 else ''})")

show_row(results, titles, figsize=(15, 5))
print("Lower threshold → more detections but more false positives.")
print("Higher threshold → fewer but more confident detections.")

In [ ]:
# 3.4 Face blur / anonymisation
def blur_faces(img, faces, blur_ksize=51):
    out = img.copy()
    if faces is None:
        return out
    for face in faces:
        x, y, fw, fh = int(face[0]), int(face[1]), int(face[2]), int(face[3])
        x, y = max(0, x), max(0, y)
        roi = out[y:y+fh, x:x+fw]
        if roi.size == 0:
            continue
        blurred_roi = cv2.GaussianBlur(roi, (blur_ksize, blur_ksize), 30)
        out[y:y+fh, x:x+fw] = blurred_roi
    return out

orig_face = cv2.imread(os.path.join(IMG_DIR, "person2.jpg"))
faces_orig = detect_faces(orig_face, yunet_path)
blurred_out = blur_faces(orig_face, faces_orig)
show_row([orig_face, blurred_out], ["Original", "Faces Blurred (Anonymised)"], figsize=(12, 5))

### 3.5 YuNet — How It Works

**YuNet** is a lightweight CNN-based face detector included in OpenCV Zoo. Key design points:

- **Architecture:** A shallow MobileNet-style backbone → detection heads at multiple scales → 5-point landmark regression (right eye, left eye, nose, right/left mouth corners)
- **Score threshold:** Controls the minimum confidence for a detection to be kept. Lower → more detections (risk: false positives on objects that look face-like). Higher → fewer but more confident detections.
- **NMS (Non-Max Suppression):** Removes overlapping duplicate boxes — keeps the highest-confidence box per face.
- **Why CNN over classical (Haar/HOG):** CNNs learn hierarchical features end-to-end; far more robust to scale, lighting, and partial occlusion than hand-crafted feature detectors.

**Trade-off:** `score_threshold=0.5` catches more faces in poor light; `0.9` reduces false alarms in cluttered scenes.

## Task 4: Object Detection with YOLOv8

**Objective:** Load YOLOv8-Nano pretrained on COCO (80 classes). Run inference on static images. Experiment with confidence and IoU thresholds. Plot a per-class detection bar chart.

**Expected output:** Annotated YOLO detections, threshold experiment, per-class bar chart.

In [ ]:
# 4.1 Load YOLOv8n (auto-downloads weights on first run)
from ultralytics import YOLO
import time

yolo_model = YOLO("yolov8n.pt")
print("YOLOv8n loaded. Classes:", len(yolo_model.names))
print("Sample classes:", {k: v for k, v in list(yolo_model.names.items())[:10]})

In [ ]:
# 4.2 Run YOLOv8 on static images
test_images = ["city_street.jpg", "animal_dog.jpg", "objects_indoor.jpg", "road.jpg"]

for fname in test_images:
    path = os.path.join(IMG_DIR, fname)
    results = yolo_model(path, conf=0.35, iou=0.45, verbose=False)
    annotated = results[0].plot()
    n_det = len(results[0].boxes)
    show(annotated, f"{fname} — {n_det} detection(s)")

# Save YOLO result
res = yolo_model(os.path.join(IMG_DIR, "city_street.jpg"), conf=0.35, verbose=False)
cv2.imwrite(os.path.join(PLOT_DIR, "yolo_detections.png"), res[0].plot())
print("Saved yolo_detections.png")

In [ ]:
# 4.3 Confidence & IoU threshold experiment
exp_img_path = os.path.join(IMG_DIR, "city_street.jpg")

conf_vals  = [0.2, 0.5, 0.8]
iou_vals   = [0.3, 0.5, 0.7]

# Vary confidence (fixed iou=0.5)
conf_results, conf_titles = [], []
for c in conf_vals:
    r = yolo_model(exp_img_path, conf=c, iou=0.5, verbose=False)
    conf_results.append(r[0].plot())
    conf_titles.append(f"conf={c}  ({len(r[0].boxes)} boxes)")
show_row(conf_results, conf_titles, figsize=(15, 5))
print("Lower conf → more boxes (more false positives); Higher conf → fewer, surer boxes.")

# Vary IoU (fixed conf=0.35) — show duplicate-box effect
iou_results, iou_titles = [], []
for iou_v in iou_vals:
    r = yolo_model(exp_img_path, conf=0.25, iou=iou_v, verbose=False)
    iou_results.append(r[0].plot())
    iou_titles.append(f"iou={iou_v}  ({len(r[0].boxes)} boxes)")
show_row(iou_results, iou_titles, figsize=(15, 5))
print("Lower IoU threshold → NMS removes more overlapping boxes (fewer duplicates).")
print("Higher IoU threshold → NMS is lenient → duplicate overlapping boxes may survive.")

In [ ]:
# 4.4 Per-class detection summary across all test images
from collections import Counter

all_images = [os.path.join(IMG_DIR, f) for f in os.listdir(IMG_DIR) if f.endswith(('.jpg', '.png'))]
class_counts = Counter()

for img_path in all_images:
    r = yolo_model(img_path, conf=0.35, iou=0.45, verbose=False)
    for box in r[0].boxes:
        class_id = int(box.cls)
        class_name = yolo_model.names[class_id]
        class_counts[class_name] += 1

if class_counts:
    classes, counts = zip(*sorted(class_counts.items(), key=lambda x: -x[1]))
    plt.figure(figsize=(max(10, len(classes)*0.8), 5))
    bars = plt.bar(classes, counts, color='steelblue', edgecolor='black')
    plt.bar_label(bars)
    plt.title("YOLOv8n — Per-Class Detection Count (All Test Images)", fontsize=13)
    plt.xlabel("COCO Class"); plt.ylabel("Detection Count")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "class_detection_bar.png"), dpi=120, bbox_inches='tight')
    plt.show()
    print("Detected classes:", dict(class_counts))
else:
    print("No detections found across test images.")

### 4.5 Why YOLO Generalises — Single-Stage Grid Design

**How YOLOv8 works:**
1. **Grid division:** The input image is divided into an S×S grid. Each cell is responsible for predicting objects whose centres fall within it.
2. **Single forward pass:** Each grid cell directly regresses: bounding box coordinates (x, y, w, h), objectness score, and class probabilities — all in one pass.
3. **Non-Max Suppression (NMS):** After prediction, overlapping boxes for the same object are pruned: keep the box with the highest confidence; suppress all others that overlap with IoU > threshold.

**Model size trade-off:**

| Variant | Params | mAP (COCO) | Speed | Use Case |
|---------|--------|-----------|-------|----------|
| yolov8**n** (Nano) | 3.2M | ~37.3 | Fastest | Edge / mobile |
| yolov8**s** (Small) | 11.2M | ~44.9 | Fast | Raspberry Pi |
| yolov8**m** (Medium) | 25.9M | ~50.2 | Moderate | Server |
| yolov8**l/x** | 43–68M | ~52–53 | Slow | Cloud |

**Why 80 classes without per-class tuning:** Trained end-to-end on COCO — the backbone learns shared visual features (edges, textures, shapes) useful across all classes; only the final prediction head is class-specific.

## Task 5: Integrated Pipeline & Final Comparison

**Objective:** Combine YuNet face detection + YOLOv8 object detection on static images (simulating a real-time pipeline). Apply morphological pre-cleaning. Benchmark FPS per component. Build a final comparison table.

**Expected output:** Combined annotated frames, FPS bar chart, comparison table.

In [ ]:
# 5.1 Unified pipeline: YuNet (green) + YOLO (red) on the same frame
def run_pipeline(img_path, yunet_path, yolo_model, score_thresh=0.7, yolo_conf=0.35):
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    out = img.copy()

    # (a) YuNet face detection — green boxes
    detector = cv2.FaceDetectorYN.create(yunet_path, "", (w, h),
                                          score_threshold=score_thresh, nms_threshold=0.3)
    _, faces = detector.detect(img)
    if faces is not None:
        for face in faces:
            x, y, fw, fh, conf = int(face[0]), int(face[1]), int(face[2]), int(face[3]), face[14]
            cv2.rectangle(out, (x, y), (x+fw, y+fh), (0, 255, 0), 2)
            cv2.putText(out, f"Face {conf:.2f}", (x, y-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    # (b) YOLO detection — red boxes with class labels
    results = yolo_model(img_path, conf=yolo_conf, iou=0.45, verbose=False)
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls_name = yolo_model.names[int(box.cls)]
        conf = float(box.conf)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(out, f"{cls_name} {conf:.2f}", (x1, y1-6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    n_faces = len(faces) if faces is not None else 0
    n_obj   = len(results[0].boxes)
    return out, n_faces, n_obj

# Run on multiple images
pipeline_images = ["person2.jpg", "city_street.jpg", "objects_indoor.jpg"]
for fname in pipeline_images:
    out_img, nf, no = run_pipeline(
        os.path.join(IMG_DIR, fname), yunet_path, yolo_model
    )
    show(out_img, f"{fname} — {nf} face(s) [green] + {no} object(s) [red]")

# Save pipeline result
out_save, _, _ = run_pipeline(os.path.join(IMG_DIR, "person2.jpg"), yunet_path, yolo_model)
cv2.imwrite(os.path.join(PLOT_DIR, "integrated_pipeline.png"), out_save)
print("Saved integrated_pipeline.png")

In [ ]:
# 5.2 Morphological pre-cleaning before detection
test_img_path = os.path.join(IMG_DIR, "city_street.jpg")
raw = cv2.imread(test_img_path)

# Add Gaussian noise to simulate a noisy camera frame
noisy_raw = raw.copy().astype(np.float32)
noisy_raw += np.random.normal(0, 25, noisy_raw.shape)
noisy_raw = np.clip(noisy_raw, 0, 255).astype(np.uint8)

# Apply morphological opening to denoise
k3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
cleaned = cv2.morphologyEx(noisy_raw, cv2.MORPH_OPEN, k3)

# Save cleaned image temporarily for YOLO
noisy_path  = os.path.join(IMG_DIR, "_tmp_noisy.jpg")
cleaned_path = os.path.join(IMG_DIR, "_tmp_cleaned.jpg")
cv2.imwrite(noisy_path,  noisy_raw)
cv2.imwrite(cleaned_path, cleaned)

r_noisy   = yolo_model(noisy_path,   conf=0.35, verbose=False)
r_cleaned = yolo_model(cleaned_path, conf=0.35, verbose=False)

show_row(
    [r_noisy[0].plot(), r_cleaned[0].plot()],
    [f"Noisy frame ({len(r_noisy[0].boxes)} detections)",
     f"After Opening ({len(r_cleaned[0].boxes)} detections)"],
    figsize=(14, 5)
)

# Cleanup temp files
os.remove(noisy_path); os.remove(cleaned_path)
print("Morphological pre-cleaning can recover detections lost to noise.")

In [ ]:
# 5.3 FPS benchmarking on static images (simulated pipeline timing)
N_FRAMES = 20
test_frame = cv2.imread(os.path.join(IMG_DIR, "person2.jpg"))
th, tw = test_frame.shape[:2]
tmp_path = os.path.join(IMG_DIR, "_bench.jpg")
cv2.imwrite(tmp_path, test_frame)

# --- Face detection FPS ---
detector_bench = cv2.FaceDetectorYN.create(yunet_path, "", (tw, th), 0.7, 0.3)
t0 = time.time()
for _ in range(N_FRAMES):
    detector_bench.detect(test_frame)
fps_face = N_FRAMES / (time.time() - t0)

# --- YOLO FPS ---
t0 = time.time()
for _ in range(N_FRAMES):
    yolo_model(tmp_path, conf=0.35, verbose=False)
fps_yolo = N_FRAMES / (time.time() - t0)

# --- Combined FPS ---
t0 = time.time()
for _ in range(N_FRAMES):
    detector_bench.detect(test_frame)
    yolo_model(tmp_path, conf=0.35, verbose=False)
fps_combined = N_FRAMES / (time.time() - t0)

os.remove(tmp_path)

labels = ["YuNet (Face)", "YOLOv8n (Objects)", "Combined Pipeline"]
fps_vals = [fps_face, fps_yolo, fps_combined]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, fps_vals, color=['limegreen', 'tomato', 'steelblue'], edgecolor='black', width=0.5)
plt.bar_label(bars, fmt='%.1f fps', padding=4, fontsize=11)
plt.ylabel("Frames per Second (FPS)", fontsize=11)
plt.title("FPS Benchmark: Face vs YOLO vs Combined", fontsize=13)
plt.ylim(0, max(fps_vals) * 1.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "fps_comparison.png"), dpi=120, bbox_inches='tight')
plt.show()

print(f"YuNet FPS: {fps_face:.1f}")
print(f"YOLOv8n FPS: {fps_yolo:.1f}")
print(f"Combined FPS: {fps_combined:.1f}")

In [ ]:
# 5.4 Final results comparison table (rendered as image for the plots/ folder)
import matplotlib.patches as mpatches

table_data = [
    ["Morphology\n(Erosion/Dilation)", "Classical", "Noise/blob shapes", "N/A", "Very High\n(>500 fps)", "Pre-process mask\nbefore detection"],
    ["Bitwise Ops\n& Histograms",       "Classical", "ROI masking\ncolour analysis", "N/A", "Very High\n(>500 fps)", "Image segmentation\nbrightness correction"],
    ["YuNet Face\nDetection",           "Deep Learning\n(CNN)", "Human faces\n+ landmarks", "Good",  f"{fps_face:.0f} fps", "Surveillance\nattendance systems"],
    ["YOLOv8n Object\nDetection",       "Deep Learning\n(CNN)", "80 COCO classes\n(people, cars…)", "Very Good", f"{fps_yolo:.0f} fps", "Retail analytics\nrobot navigation"],
    ["Integrated\nPipeline",            "Classical + DL", "Faces + Objects\n(combined)", "Very Good", f"{fps_combined:.0f} fps", "Smart surveillance\nedge AI systems"],
]
col_headers = ["Technique", "Type", "What It Detects", "Lighting\nRobustness", "Approx. FPS", "Best Use Case"]

fig, ax = plt.subplots(figsize=(16, 5))
ax.axis('off')
tbl = ax.table(
    cellText=table_data,
    colLabels=col_headers,
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 2.5)

# Style header row
for j in range(len(col_headers)):
    tbl[(0, j)].set_facecolor('#2b5fad')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')
# Alternate row colours
for i in range(1, len(table_data)+1):
    for j in range(len(col_headers)):
        tbl[(i, j)].set_facecolor('#f0f4ff' if i % 2 == 0 else 'white')

plt.title("Final Comparison: All Techniques", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "results_table.png"), dpi=120, bbox_inches='tight')
plt.show()
print("Saved results_table.png")

### 5.5 Closing Reflection

**Best combination for real-time monitoring:**  
Classical morphological opening (3×3 ellipse kernel) as a pre-processing step + YuNet (score_threshold=0.7) for faces + YOLOv8n (conf=0.4) for objects gives the best **accuracy-to-compute ratio**. The classical pre-processing is essentially free (>500 fps) and measurably reduces false detections in noisy frames.

**Edge device deployment adjustments:**
- Swap YOLOv8n for a quantised INT8 version (using `yolo export format=onnx` + OpenVINO/TFLite runtime) to cut inference time by ~3×.
- Raise `score_threshold` to 0.85 to reduce false positives (edge devices can't afford to process many ROIs).
- Use a smaller input resolution (e.g. 320×240) to halve YOLO inference time at slight accuracy cost.

**Cloud server adjustments:**
- Upgrade to YOLOv8m or YOLOv8l for higher mAP (50+ on COCO).
- Lower `score_threshold` to 0.5 to catch borderline detections — the server can handle the extra post-processing.
- Run both detectors on every frame rather than every Nth frame.

**Key insight:** Classical CV (morphology, histograms) and deep learning are not competing approaches — they are **complementary layers** in a production pipeline, each handling the part of the problem it is best at.

In [ ]:
# ============================================================
# OPTIONAL — Real-time Webcam Functions
# (not run automatically — call manually if webcam available)
# ============================================================

def webcam_face_detection(model_path, score_thresh=0.7, max_frames=300):
    """Real-time YuNet face detection on webcam. Press Q to quit."""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Webcam not available. Skipping.")
        return
    fps_list = []
    frame_count = 0
    while frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        h, w = frame.shape[:2]
        det = cv2.FaceDetectorYN.create(model_path, "", (w, h),
                                         score_threshold=score_thresh, nms_threshold=0.3)
        t0 = time.time()
        _, faces = det.detect(frame)
        fps_list.append(1 / (time.time() - t0 + 1e-9))
        out = draw_faces(frame, faces)
        cv2.putText(out, f"FPS:{fps_list[-1]:.1f}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
        cv2.imshow("YuNet Face Detection — Q to quit", out)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        frame_count += 1
    cap.release(); cv2.destroyAllWindows()
    print(f"Average FPS (YuNet): {np.mean(fps_list):.1f}")

def webcam_yolo_detection(yolo_model, conf=0.35, max_frames=300):
    """Real-time YOLOv8 detection on webcam. Press Q to quit."""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Webcam not available. Skipping.")
        return
    fps_list = []
    frame_count = 0
    while frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        t0 = time.time()
        results = yolo_model(frame, conf=conf, iou=0.45, verbose=False)
        fps_list.append(1 / (time.time() - t0 + 1e-9))
        out = results[0].plot()
        cv2.putText(out, f"FPS:{fps_list[-1]:.1f}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
        cv2.imshow("YOLOv8 Detection — Q to quit", out)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        frame_count += 1
    cap.release(); cv2.destroyAllWindows()
    print(f"Average FPS (YOLO): {np.mean(fps_list):.1f}")

def webcam_integrated_pipeline(yunet_path, yolo_model, max_frames=300):
    """Integrated pipeline: YuNet (green) + YOLO (red) on webcam."""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Webcam not available. Skipping.")
        return
    fps_list = []
    frame_count = 0
    while frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        h, w = frame.shape[:2]
        det = cv2.FaceDetectorYN.create(yunet_path, "", (w, h), 0.7, 0.3)
        t0 = time.time()
        _, faces = det.detect(frame)
        results = yolo_model(frame, conf=0.35, iou=0.45, verbose=False)
        fps_list.append(1 / (time.time() - t0 + 1e-9))
        out = frame.copy()
        if faces is not None:
            for face in faces:
                x, y, fw, fh = int(face[0]), int(face[1]), int(face[2]), int(face[3])
                cv2.rectangle(out, (x,y), (x+fw, y+fh), (0,255,0), 2)
        for box in results[0].boxes:
            x1,y1,x2,y2 = map(int, box.xyxy[0])
            cv2.rectangle(out, (x1,y1),(x2,y2),(0,0,255),2)
            cv2.putText(out, yolo_model.names[int(box.cls)], (x1,y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)
        cv2.putText(out, f"FPS:{fps_list[-1]:.1f}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,0), 2)
        cv2.imshow("Integrated Pipeline — Q to quit", out)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        frame_count += 1
    cap.release(); cv2.destroyAllWindows()
    print(f"Average FPS (Combined): {np.mean(fps_list):.1f}")

print("Webcam functions defined. Call them manually:")
print("  webcam_face_detection(yunet_path)")
print("  webcam_yolo_detection(yolo_model)")
print("  webcam_integrated_pipeline(yunet_path, yolo_model)")